# Homework №2 | Practice

## <a href=https://arxiv.org/abs/2206.00927>DPM-solver</a> for flow matching (7 pt)

In this notebook, you are asked to implement three numerical solvers for flow matching: `Euler`, `DPM-1`, and `DPM-2`, and evaluate them under different timestep schedules (`linear`, `EDM`, and `SD3`). In addition, you will be asked to modify `DPM-2` by adding a multistep mode (a two-step <a href=https://en.wikipedia.org/wiki/Linear_multistep_method> Adams linear multistep method</a>).

First, let's load the pipeline of the <a href=https://github.com/NVlabs/Sana/tree/main>SANA</a> model, which is trained for text-to-image generation at 1024x1024. This model is trained in the *flow matching* setting and predicts the *velocity field* $v_{\theta}(x_{t}, t)$.

*Forward process:* $x_{t} = (1 - t) x_{0} + t \epsilon$

In [ ]:
!pip install -U diffusers --upgrade

In [ ]:
import torch
from functools import partial
from diffusers import SanaPipeline


pipe = SanaPipeline.from_pretrained(
    "Efficient-Large-Model/Sana_600M_1024px_diffusers",
    variant="fp16",
    torch_dtype=torch.float16,
)
pipe.to("cuda")

### Add <a href=https://arxiv.org/abs/2207.12598> Classifier-Free Guidance</a> to SANA sampling

Below, you are given a sampling function for the SANA model. Update `_v_pred_fn` to support CFG, implemented using **a single model forward pass**.


In [ ]:
from tqdm.auto import tqdm

@torch.no_grad()
def _v_pred_fn(
    latents, 
    sigma,
    prompt_embeds, prompt_attention_mask,
    do_classifier_free_guidance, guidance_scale,
    pipe
):
    if do_classifier_free_guidance:
        # YOUR CODE
        pass
    else:
        latent_model_input = latents
        
    latent_model_input = latent_model_input.to(prompt_embeds.dtype)
    
    timestep = (sigma * 1000).to(int)
    timestep = timestep.expand(latent_model_input.shape[0])

    v_pred = pipe.transformer(
        latent_model_input,
        encoder_hidden_states=prompt_embeds,
        encoder_attention_mask=prompt_attention_mask,
        timestep=timestep,
        return_dict=False,
    )[0]

    # perform guidance
    if do_classifier_free_guidance:
        # YOUR CODE
        pass
        
    pipe.n_steps += 1
    return v_pred


@torch.no_grad()
def run_inference(
    pipe,
    step_fn, # solver step function
    sigmas,  # sigmas defined my your timestep scheduler
    prompt,
    negative_prompt = "", # Prompt used for CFG
    guidance_scale = 4.5,
    device="cuda",
    seed = 0,
    # Instruction prompt for Gemma2. Just keep it as it
    complex_human_instruction = [ 
        "Given a user prompt, generate an 'Enhanced prompt' that provides detailed visual descriptions suitable for image generation. Evaluate the level of detail in the user prompt:",
        "- If the prompt is simple, focus on adding specifics about colors, shapes, sizes, textures, and spatial relationships to create vivid and concrete scenes.",
        "- If the prompt is already detailed, refine and enhance the existing details slightly without overcomplicating.",
        "Here are examples of how to transform or refine prompts:",
        "- User Prompt: A cat sleeping -> Enhanced: A small, fluffy white cat curled up in a round shape, sleeping peacefully on a warm sunny windowsill, surrounded by pots of blooming red flowers.",
        "- User Prompt: A busy city street -> Enhanced: A bustling city street scene at dusk, featuring glowing street lamps, a diverse crowd of people in colorful clothing, and a double-decker bus passing by towering glass skyscrapers.",
        "Please generate only the enhanced description for the prompt below and avoid including any additional commentary or evaluations:",
        "User Prompt: ",
    ],
):
    """
    Main sampling function. May be useful for understanding and debugging.
    """
    do_classifier_free_guidance = guidance_scale > 1.0
    generator = torch.Generator(device=device).manual_seed(seed)
    
    # 3. Encode input prompt
    (
        prompt_embeds,
        prompt_attention_mask,
        uncond_prompt_embeds, # null prompt embeds for CFG
        uncond_prompt_attention_mask,
    ) = pipe.encode_prompt(
        prompt,
        do_classifier_free_guidance,
        negative_prompt=negative_prompt,
        device=device,
        complex_human_instruction=complex_human_instruction,
    )
    
    # If CFG, we concat uncond_prompt_embeds and prompt_embeds along the batch axis to avoid double forward passes 
    if do_classifier_free_guidance:
        prompt_embeds = torch.cat([uncond_prompt_embeds, prompt_embeds], dim=0)
        prompt_attention_mask = torch.cat([uncond_prompt_attention_mask, prompt_attention_mask], dim=0)

    # 4. Preprocess sigmas
    sigmas = torch.tensor(sigmas).to(device)

    # 5. Prepare latents.
    latent_channels = pipe.transformer.config.in_channels
    latent_shape = (1, latent_channels, 32, 32)
    latents = torch.randn(
        *latent_shape,
        generator=generator,
        device=device
    )
    
    # Predefine args for v_pred function for convenient use in step_fn
    v_pred_fn = partial(
        _v_pred_fn,
        prompt_embeds=prompt_embeds,
        prompt_attention_mask=prompt_attention_mask,
        do_classifier_free_guidance=do_classifier_free_guidance,
        guidance_scale=guidance_scale,
        pipe=pipe,
    )

    # 6. Denoising loop
    prev_eps_pred = None
    pipe.n_steps = 0

    for i, sigma in enumerate(tqdm(sigmas[:-1])):
        # Predict v for x_t
        v_pred = v_pred_fn(latents, sigma)
        
        # make a step: x_t -> x_t-1
        latents = step_fn(
            latents, sigmas, v_pred, i,
            v_pred_fn=v_pred_fn, # you will need this for DPM-2 single
            prev_eps_pred=prev_eps_pred, # you will need this for DPM-2 multi
        )
        prev_eps_pred = <YOUR CODE>

    # Decode latents to images
    latents = latents.to(pipe.vae.dtype)
    images = pipe.vae.decode(latents / pipe.vae.config.scaling_factor, return_dict=False)[0]            
    images = pipe.image_processor.postprocess(images, output_type='pil')
    print(f'The number of neural network evaluations (NFE) is equal to {pipe.n_steps}.')
    return images

### Implement Solvers and Timestep Schedulers

You are asked to implement two functions: `step_fn` and `schedule_fn`.

### Solvers

The function `step_fn` performs a single integration step using one of the following solvers:

- `euler`
- `dpm1_single`
- `dpm2_single`
- `dpm2_multi`

Here, `single` and `multi` denote the single-step and multistep solver families discussed in class.

**Euler** must be implemented using the velocity-prediction (v-prediction) parameterization for solving the PF-ODE: $dx = v(x,t)dt$

**DPM-Solvers** must be implemented using the noise-prediction ($\epsilon$-prediction) parameterization, following the derivations presented in theory.


### Schedulers

The function `schedule_fn` defines the sampling schedule in terms of time $t$. You are asked to implement three commonly used schedules:

- `linear`
- `edm`
- `sd3`

We follow the standard general notation and denote $t$ as $\sigma$, where: $\sigma \in [1, 0]$.

---

#### <a href=https://github.com/NVlabs/edm/tree/main> EDM</a> Schedule

$ \sigma_{i < N} = \left(\sigma_{max} ^ {\frac{1}{\rho}} + \frac{i}{N-1} \cdot \left(\sigma_{min} ^ {\frac{1}{\rho}} - \sigma_{max} ^ {\frac{1}{\rho}} \right) \right) ^ {\rho}, \sigma_{N} = 0$,

Where:

- $N$ — number of generation steps  
- $\rho$ — hyperparameter (typically \(7\))  
- $\sigma_{\max}$, $\sigma_{\min}$ — maximum and minimum time values  

EDM is commonly used in the VE setting with:


$\sigma_{\max} = 80, \quad \sigma_{\min} = 0.002.$


For flow matching, $\sigma_{\max} = 1.0$. However, we recommend using $\sigma_{\max} = 80$ and then rescaling $\sigma$ to $[1, 0]$.

---

####  <a href=https://arxiv.org/pdf/2403.03206v1>SD3</a> Schedule

$\sigma_i =
\frac{s \cdot i}{1 + (s - 1)i},
\quad i \in [0,1]$,

where:

- $s \ge 1$ controls the shift of the schedule toward higher noise levels  
- $s = 3$ is the default value used in SD3 models  

In [ ]:
def step_fn(latents, sigmas, v_pred, i, solver_name, v_pred_fn, prev_eps_pred=None):
    '''
    Recommendations:

    0. sigmas <=> t --> sigmas in [1, 0]. sigmas[i] > sigmas[i+1]

    1. For extra model predictions use v_pred_fn. 

    2. Pay attention to numerical instabilities at endpoints

    3. prev_eps_pred corresponds to eps_pred at the previous step. None for the first step
    '''
        
    if solver_name == 'euler':
        # YOUR CODE
        pass
        
    elif "dpm" in solver_name:
        # Define eps_pred for DPM solver implementation
        # eps_pred = <YOUR CODE>
    
        if solver_name == 'dpm1_single':
            # YOUR CODE
            pass

        elif solver_name == 'dpm2_single':
            # YOUR CODE
            pass

        elif solver_name == 'dpm2_multi':
            # YOUR CODE
            pass
    else:
        raise ValueError

    return latents


def schedule_fn(n_points, scheduler_name, sd3_s=3, edm_rho=7):
    n_points = n_points + 1 # an additional point for t = 0
    
    if scheduler_name == 'linear':
        # YOUR CODE
        pass
        
    elif scheduler_name == 'sd3':
        s = sd3_s
        
        # YOUR CODE
        pass
        
    elif scheduler_name == 'edm':
        rho = edm_rho
        sigma_max = 80
        sigma_min = 0.002
        
        # YOUR CODE
        pass
        
    return sigmas

After implementing these functions, test them using the cell below. You can experiment with the number of steps and other hyperparameters. For reference, we provide our results below:

<div>
<img src="https://i.postimg.cc/HLPMjd9n/hw2-examples.jpg"/ style="width:800px; height:auto; vertical-align:middle;">
</div>

Note that you **do not** have to obtain the same images. The example provides the reference quality, we expect to see.

In [ ]:
import numpy as np
from functools import partial

image = run_inference(
    pipe,
    step_fn=partial(step_fn, solver_name='euler'),  # Play with it
    sigmas=schedule_fn(n_points=20, scheduler_name='linear'),  # Play with it

    prompt="a cat",
    guidance_scale=4.5,
    seed=42,
)[0]
image.resize((512, 512)) 

### Prompt exaples

For your experiments, you may use any prompts of your choice. A few example prompts are provided below for reference:


`"Self-portrait oil painting, a beautiful cyborg with golden hair, 8k"`

`"A girl with pale blue hair and a cami tank top"`

`"Four cows in a pen on a sunny day"`

`"Three dogs sleeping together on an unmade bed"`

`"A sky blue colored hippopotamus"`

`"The interior of a mad scientists laboratory, Cluttered with science experiments, tools and strange machine, Eerie purple light, Close up, by Miyazaki"`

`"a barred owl peeking out from dense tree branches"`

`"a close-up of a blue dragonfly on a daffodil"`

`"A green train is coming down the tracks"`

`"A photograph of the inside of a subway train. There are frogs sitting on the seats. One of them is reading a newspaper. The window shows the river in the background."`

`"A high resolution photo of a donkey in a clown costume giving a lecture at the front of a lecture hall. The blackboard has mathematical equations on it. There are many students in the lecture hall."`

`"A raccoon wearing formal clothes, wearing a tophat and holding a cane. The raccoon is holding a garbage bag. Oil painting in the style of abstract cubism."`

### Analyse the model performance with CFG (1 pt)

* Using the Euler solver, compare the model performance for different CFG scales 
* Discuss the observed behavior at low and high guidance scales


In [ ]:
<YOUR EXPERIMENTS AND CONCLUSIONS>

### Compare timestep schedulers (2 pt)

* Using the Euler solver, compare the model performance for Linear, EDM, SD3 schedules for 20 steps (please support your conclusion with image examples)
* Visualize the schedules and try to explain why some schedulers are better than others

**PS:** feel free to use any other solvers and number of steps if you’d like. :)    

In [ ]:
<YOUR EXPERIMENTS AND CONCLUSIONS>

### Compare DPM-1 and Euler solvers (1 pt)

* Compare them only for the best scheduler from above

In [ ]:
<YOUR EXPERIMENTS AND CONCLUSIONS>

### Compare DPM-2 single and DPM-1 (2 pt)

Compare `DPM-2 single` and `DPM-1` solvers under the same NFE budget.

* Compare them only for the best scheduler from above
* Evaluate several NFE budgets. For each budget, determine the number of solver steps required by each method to meet that budget.
* Compare performance at each budget and discuss which solver performs better in your opinion.

**PS:** remember that NFE != number of steps

In [ ]:
<YOUR EXPERIMENTS AND CONCLUSIONS>

### Compare DPM-2 single vs DPM-2 multi (2 pt)

Compare `DPM-2 single` and `DPM-2 multi` solvers under the same NFE budget.

Follow the instructions provided in the previous cell

In [ ]:
<YOUR EXPERIMENTS AND CONCLUSIONS>